# Yield Mixed-Effects Models and Figures

This notebook computes paired-difference plots and mixed-effects model diagnostics.

It only writes figure files (no extra CSV/MD outputs) to keep the results directory minimal.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# Resolve project root whether notebook runs from repo root or notebooks/
ROOT = Path.cwd().resolve()
if not (ROOT / "cleaned_data").exists() and (ROOT.parent / "cleaned_data").exists():
    ROOT = ROOT.parent

CLEAN_DIR = ROOT / "cleaned_data"
RESULT_DIR = ROOT / "results" / "yield"
FIG_DIR = RESULT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

LT_PATH = CLEAN_DIR / "longterm_yield_analysis_locked.csv"
ACUTE_PATH = CLEAN_DIR / "acute_yield_analysis_locked.csv"

print("ROOT:", ROOT)
print("LT_PATH exists:", LT_PATH.exists())
print("ACUTE_PATH exists:", ACUTE_PATH.exists())


In [ ]:
lt = pd.read_csv(LT_PATH)
acute = pd.read_csv(ACUTE_PATH)

lt["include_in_yield_model"] = lt["include_in_yield_model"].astype(int)
acute["include_in_yield_model"] = acute["include_in_yield_model"].astype(int)

lt_keep = lt[lt["include_in_yield_model"] == 1].copy()
acute_keep = acute[acute["include_in_yield_model"] == 1].copy()

# Long-term paired differences within cultivar x set
lt_wide = lt_keep.pivot(index=["cultivar", "set_id"], columns="heat_trt", values=[
    "healthy_weight_g", "pct_rotten_count", "pct_rotten_weight"
]).reset_index()
lt_wide.columns = [f"{a}_{b.lower()}" if isinstance(a, str) and b else a for a, b in lt_wide.columns]
lt_wide = lt_wide.rename(columns={
    "healthy_weight_g_control": "healthy_weight_control",
    "healthy_weight_g_otc": "healthy_weight_heat",
    "pct_rotten_count_control": "pct_rotten_count_control",
    "pct_rotten_count_otc": "pct_rotten_count_heat",
    "pct_rotten_weight_control": "pct_rotten_weight_control",
    "pct_rotten_weight_otc": "pct_rotten_weight_heat",
})
lt_wide["healthy_weight_diff"] = lt_wide["healthy_weight_heat"] - lt_wide["healthy_weight_control"]
lt_wide["pct_rotten_count_diff"] = lt_wide["pct_rotten_count_heat"] - lt_wide["pct_rotten_count_control"]
lt_wide["pct_rotten_weight_diff"] = lt_wide["pct_rotten_weight_heat"] - lt_wide["pct_rotten_weight_control"]

# Acute paired differences within cultivar x heat_level x replicate
acute_wide = acute_keep.pivot(index=["cultivar", "heat_level", "replicate"], columns="is_control", values=[
    "healthy_weight_g", "pct_rotten_count", "pct_rotten_weight"
]).reset_index()
acute_wide.columns = [
    f"{a}_{'control' if int(b)==1 else 'heat'}" if isinstance(a, str) and b in [0, 1] else a
    for a, b in acute_wide.columns
]
acute_wide["healthy_weight_diff"] = acute_wide["healthy_weight_g_heat"] - acute_wide["healthy_weight_g_control"]
acute_wide["pct_rotten_count_diff"] = acute_wide["pct_rotten_count_heat"] - acute_wide["pct_rotten_count_control"]
acute_wide["pct_rotten_weight_diff"] = acute_wide["pct_rotten_weight_heat"] - acute_wide["pct_rotten_weight_control"]

print("Long-term pairs:", len(lt_wide))
print("Acute pairs:", len(acute_wide))
lt_wide.head()


In [ ]:
def fit_mixedlm(formula: str, data: pd.DataFrame, group_col: str):
    methods = ["lbfgs", "powell", "cg", "nm"]
    fallback = None
    errors = []
    for method in methods:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = smf.mixedlm(formula, data=data, groups=data[group_col]).fit(
                    reml=False,
                    method=method,
                    maxiter=1000,
                    disp=False,
                )
            if model.converged:
                return model, method
            if fallback is None:
                fallback = (model, method)
        except Exception as exc:
            errors.append(f"{method}: {exc}")
    if fallback is not None:
        return fallback
    raise RuntimeError("MixedLM failed for all optimizers: " + " | ".join(errors))


def _scalar(value) -> float:
    arr = np.asarray(value).astype(float).reshape(-1)
    return float(arr[0]) if arr.size else np.nan


def mixedlm_wald_terms_table(model, outcome: str, model_name: str) -> pd.DataFrame:
    table = model.wald_test_terms(skip_single=False).table.reset_index().rename(columns={"index": "term"})
    p_col = "pvalue" if "pvalue" in table.columns else "p_value"
    return pd.DataFrame(
        {
            "model_name": model_name,
            "outcome": outcome,
            "term": table["term"],
            "statistic": table["statistic"].map(_scalar),
            "df_constraint": table["df_constraint"].map(_scalar),
            "p_value": table[p_col].map(_scalar),
        }
    )


lt_model_df = lt_keep.copy()
lt_model_df["set_key"] = lt_model_df["cultivar"].astype(str) + "_" + lt_model_df["set_id"].astype(str)

lt_wald_tables = []
for out in ["healthy_weight_g", "pct_rotten_count", "pct_rotten_weight"]:
    model, optimizer = fit_mixedlm(f"{out} ~ C(cultivar) * C(heat_trt)", lt_model_df, "set_key")
    lt_wald_tables.append(mixedlm_wald_terms_table(model, out, "longterm_mixedlm"))
lt_wald = pd.concat(lt_wald_tables, ignore_index=True)

acute_model_df = acute_keep.copy()
acute_model_df["pair_id"] = (
    acute_model_df["cultivar"].astype(str)
    + "_"
    + acute_model_df["heat_level"].astype(str)
    + "_"
    + acute_model_df["replicate"].astype(str)
)

acute_wald_tables = []
for out in ["healthy_weight_g", "pct_rotten_count", "pct_rotten_weight"]:
    model, optimizer = fit_mixedlm(f"{out} ~ C(cultivar) * C(heat_level) * C(is_control)", acute_model_df, "pair_id")
    acute_wald_tables.append(mixedlm_wald_terms_table(model, out, "acute_mixedlm"))
acute_wald = pd.concat(acute_wald_tables, ignore_index=True)

print("Long-term smallest p by outcome:")
for out, sub in lt_wald[lt_wald["term"] != "Intercept"].groupby("outcome"):
    best = sub.sort_values("p_value", ascending=True).iloc[0]
    print(f"- {out}: {best['term']} (p={best['p_value']:.4g})")

print("\nAcute smallest p by outcome:")
for out, sub in acute_wald[acute_wald["term"] != "Intercept"].groupby("outcome"):
    best = sub.sort_values("p_value", ascending=True).iloc[0]
    print(f"- {out}: {best['term']} (p={best['p_value']:.4g})")

lt_wald.sort_values(["outcome", "p_value"], ascending=[True, True])


In [ ]:
# Figure 1: Long-term healthy weight diff by cultivar
fig1, ax1 = plt.subplots(figsize=(7, 4))
for i, c in enumerate(["MQ", "St"]):
    vals = lt_wide.loc[lt_wide["cultivar"] == c, "healthy_weight_diff"].to_numpy()
    x = np.full(len(vals), i + 1)
    ax1.scatter(x, vals, alpha=0.8, label=c)
    ax1.hlines(vals.mean(), i + 0.75, i + 1.25, linewidth=3)
ax1.axhline(0, color="black", linewidth=1)
ax1.set_xticks([1, 2])
ax1.set_xticklabels(["MQ", "St"])
ax1.set_ylabel("Healthy weight diff (heat - control, g)")
ax1.set_title("Long-term: Healthy Weight Paired Differences")
ax1.grid(alpha=0.2)
fig1.tight_layout()
fig1_path = FIG_DIR / "yield_longterm_healthy_weight_diff.png"
fig1.savefig(fig1_path, dpi=180)
plt.show()

# Figure 2: Long-term rotten-count proportion diff by cultivar
fig2, ax2 = plt.subplots(figsize=(7, 4))
for i, c in enumerate(["MQ", "St"]):
    vals = lt_wide.loc[lt_wide["cultivar"] == c, "pct_rotten_count_diff"].to_numpy()
    x = np.full(len(vals), i + 1)
    ax2.scatter(x, vals, alpha=0.8, label=c)
    ax2.hlines(vals.mean(), i + 0.75, i + 1.25, linewidth=3)
ax2.axhline(0, color="black", linewidth=1)
ax2.set_xticks([1, 2])
ax2.set_xticklabels(["MQ", "St"])
ax2.set_ylabel("Rotten-count proportion diff (heat - control)")
ax2.set_title("Long-term: Rotten Proportion Paired Differences")
ax2.grid(alpha=0.2)
fig2.tight_layout()
fig2_path = FIG_DIR / "yield_longterm_rotten_count_diff.png"
fig2.savefig(fig2_path, dpi=180)
plt.show()

# Figure 3: Acute healthy weight diff by heat level and cultivar
heat_levels = ["A", "B", "C", "D"]
fig3, ax3 = plt.subplots(figsize=(9, 4.5))
positions = []
data = []
labels = []
for i, h in enumerate(heat_levels):
    for j, c in enumerate(["MQ", "St"]):
        vals = acute_wide.loc[(acute_wide["heat_level"] == h) & (acute_wide["cultivar"] == c), "healthy_weight_diff"].to_numpy()
        positions.append(i * 3 + j + 1)
        data.append(vals)
        labels.append(f"{h}-{c}")
ax3.boxplot(data, positions=positions, widths=0.65, patch_artist=False)
ax3.axhline(0, color="black", linewidth=1)
ax3.set_xticks(positions)
ax3.set_xticklabels(labels, rotation=45, ha="right")
ax3.set_ylabel("Healthy weight diff (heat - control, g)")
ax3.set_title("Acute: Healthy Weight Paired Differences")
ax3.grid(alpha=0.2)
fig3.tight_layout()
fig3_path = FIG_DIR / "yield_acute_healthy_weight_diff_boxplot.png"
fig3.savefig(fig3_path, dpi=180)
plt.show()

# Figure 4: Acute rotten-count proportion diff by heat level and cultivar
fig4, ax4 = plt.subplots(figsize=(9, 4.5))
positions2 = []
data2 = []
labels2 = []
for i, h in enumerate(heat_levels):
    for j, c in enumerate(["MQ", "St"]):
        vals = acute_wide.loc[(acute_wide["heat_level"] == h) & (acute_wide["cultivar"] == c), "pct_rotten_count_diff"].to_numpy()
        positions2.append(i * 3 + j + 1)
        data2.append(vals)
        labels2.append(f"{h}-{c}")
ax4.boxplot(data2, positions=positions2, widths=0.65, patch_artist=False)
ax4.axhline(0, color="black", linewidth=1)
ax4.set_xticks(positions2)
ax4.set_xticklabels(labels2, rotation=45, ha="right")
ax4.set_ylabel("Rotten-count proportion diff (heat - control)")
ax4.set_title("Acute: Rotten Proportion Paired Differences")
ax4.grid(alpha=0.2)
fig4.tight_layout()
fig4_path = FIG_DIR / "yield_acute_rotten_count_diff_boxplot.png"
fig4.savefig(fig4_path, dpi=180)
plt.show()

print("Saved figures:")
for p in [fig1_path, fig2_path, fig3_path, fig4_path]:
    print("-", p)


In [ ]:
print("This notebook writes figure files only:")
for p in [fig1_path, fig2_path, fig3_path, fig4_path]:
    print("-", p)
print("Model coefficient/Wald CSV outputs are written by notebooks/yield_analysis.ipynb.")
